[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/SFM/blob/main/Quantlets/Ch_02/SFM_ch2_modern_extensions/SFM_ch2_modern_extensions.ipynb)

# SFM_ch2_modern_extensions

Modern Extensions: GARCH-t, Mixture Models, and Regime Switching
Description:
- GARCH(1,1) with Student-t innovations
- Gaussian mixture model for return distributions
- Regime-switching concept and hidden states
- Comparison of conditional vs unconditional distributions
Statistics of Financial Markets course

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from scipy import stats
import os
import warnings
warnings.filterwarnings('ignore')

# !pip install arch scikit-learn

In [ ]:
# Chart style settings - Nature journal quality
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 8
plt.rcParams['axes.labelsize'] = 9
plt.rcParams['axes.titlesize'] = 10
plt.rcParams['xtick.labelsize'] = 8
plt.rcParams['ytick.labelsize'] = 8
plt.rcParams['legend.fontsize'] = 8
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.5
plt.rcParams['lines.linewidth'] = 0.75

# Color palette
MAIN_BLUE = '#1A3A6E'
CRIMSON = '#DC3545'
FOREST = '#2E7D32'
AMBER = '#B5853F'
ORANGE = '#E67E22'

# Output directory
CHART_DIR = os.path.join('..', '..', '..', 'charts')
os.makedirs(CHART_DIR, exist_ok=True)

def save_fig(name):
    """Save figure with transparent background."""
    plt.savefig(os.path.join(CHART_DIR, f'{name}.pdf'),
                bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(CHART_DIR, f'{name}.png'),
                bbox_inches='tight', transparent=True, dpi=300)
    plt.close()
    print(f"   Saved: {name}.pdf/.png")

## Download Data

In [ ]:
# Download S&P 500 data
data = yf.download('^GSPC', start='2000-01-01', end='2025-01-01', progress=False)
close = data['Close'].squeeze()
log_ret = np.log(close / close.shift(1)).dropna()

print(f"   Period: {close.index[0].strftime('%Y-%m-%d')} to {close.index[-1].strftime('%Y-%m-%d')}")
print(f"   Observations: {len(log_ret)}")
print(f"   Mean: {log_ret.mean():.6f}")
print(f"   Std:  {log_ret.std():.6f}")
print(f"   Skew: {log_ret.skew():.4f}")
print(f"   Kurt: {log_ret.kurtosis():.4f}")

## GARCH(1,1) with Student-t Innovations

In [ ]:
from arch import arch_model

# Fit GARCH(1,1) with Student-t distribution
# arch expects returns in percentage form
ret_pct = log_ret * 100

am = arch_model(ret_pct, vol='Garch', p=1, q=1, dist='t', mean='Constant')
res = am.fit(disp='off')
print(res.summary())

# Extract conditional volatility and fitted nu
cond_vol = res.conditional_volatility / 100  # back to decimal
nu = res.params['nu']
print(f"\n   Fitted degrees of freedom (nu): {nu:.2f}")

# --- Plot: Returns with conditional volatility bands ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 5), sharex=True)

# Panel 1: Returns with +/- 2*sigma bands
ax1.plot(log_ret.index, log_ret.values, color=MAIN_BLUE, linewidth=0.3, alpha=0.7,
         label='S&P 500 log-returns')
ax1.plot(cond_vol.index, 2 * cond_vol.values, color=CRIMSON, linewidth=0.6,
         label='$\\pm 2\\sigma_t$ (GARCH-t)')
ax1.plot(cond_vol.index, -2 * cond_vol.values, color=CRIMSON, linewidth=0.6)
ax1.axhline(0, color='gray', linewidth=0.3)
ax1.set_ylabel('Log-return')
ax1.set_title('GARCH(1,1)-t: Returns with conditional volatility bands', fontweight='bold', fontsize=9)
ax1.legend(loc='upper right', fontsize=7, frameon=False)
ax1.text(0.02, 0.05, f'$\\nu$ = {nu:.2f}', transform=ax1.transAxes,
         fontsize=8, color=CRIMSON, va='bottom')

# Panel 2: Conditional volatility time series
ax2.plot(cond_vol.index, cond_vol.values * 100, color=FOREST, linewidth=0.6)
ax2.fill_between(cond_vol.index, 0, cond_vol.values * 100, color=FOREST, alpha=0.15)
ax2.set_ylabel('Conditional volatility (%)')
ax2.set_xlabel('Date')
ax2.set_title('GARCH(1,1)-t: Conditional volatility $\\sigma_t$', fontweight='bold', fontsize=9)

plt.tight_layout()
save_fig('ch2_garch_t_fit')

## Standardized Residuals

In [ ]:
# Extract standardized residuals
std_resid = res.std_resid

fig, axes = plt.subplots(1, 3, figsize=(10, 3))

# --- Panel 1: Histogram of standardized residuals + fitted Student-t ---
ax = axes[0]
ax.hist(std_resid, bins=100, density=True, alpha=0.4, color=MAIN_BLUE,
        edgecolor='none', label='Std. residuals')
x_grid = np.linspace(-6, 6, 300)
# Fit Student-t to standardized residuals
t_params_resid = stats.t.fit(std_resid)
ax.plot(x_grid, stats.t.pdf(x_grid, *t_params_resid), color=CRIMSON, linewidth=1.0,
        label=f'Student-t ($\\nu$={t_params_resid[0]:.1f})')
ax.plot(x_grid, stats.norm.pdf(x_grid), color=FOREST, linewidth=0.8,
        linestyle='--', label='Normal')
ax.set_xlabel('Standardized residual')
ax.set_ylabel('Density')
ax.set_title('GARCH-t std. residuals', fontweight='bold', fontsize=9)
ax.legend(fontsize=6, frameon=False)

# --- Panel 2: QQ-plot of standardized residuals ---
ax = axes[1]
theoretical_q = stats.probplot(std_resid, dist='norm')
theor_quantiles = theoretical_q[0][0]
sample_quantiles = theoretical_q[0][1]
ax.scatter(theor_quantiles, sample_quantiles, s=2, alpha=0.4, color=CRIMSON,
           edgecolors='none')
q_min, q_max = theor_quantiles.min(), theor_quantiles.max()
ax.plot([q_min, q_max], [q_min, q_max], '--', color='gray', linewidth=0.8)
ax.set_xlabel('Theoretical Quantiles (Normal)')
ax.set_ylabel('Sample Quantiles')
ax.set_title('QQ-plot: std. residuals', fontweight='bold', fontsize=9)

# --- Panel 3: Compare raw returns vs standardized residuals ---
ax = axes[2]
z_raw = (log_ret - log_ret.mean()) / log_ret.std()
ax.hist(z_raw, bins=100, density=True, alpha=0.3, color=AMBER,
        edgecolor='none', label='Raw returns (std.)')
ax.hist(std_resid, bins=100, density=True, alpha=0.3, color=MAIN_BLUE,
        edgecolor='none', label='GARCH std. residuals')
ax.plot(x_grid, stats.norm.pdf(x_grid), color=FOREST, linewidth=0.8,
        linestyle='--', label='Normal')
ax.set_xlabel('Standardized value')
ax.set_ylabel('Density')
ax.set_title('Raw returns vs GARCH residuals', fontweight='bold', fontsize=9)
ax.legend(fontsize=6, frameon=False)
ax.set_xlim(-6, 6)

plt.tight_layout()
save_fig('ch2_garch_residuals')

## Gaussian Mixture Model

In [ ]:
from sklearn.mixture import GaussianMixture

# Fit 2-component Gaussian mixture to S&P 500 returns
X = log_ret.values.reshape(-1, 1)
gmm = GaussianMixture(n_components=2, random_state=42, max_iter=500)
gmm.fit(X)

# Extract parameters
weights = gmm.weights_
means = gmm.means_.flatten()
stds = np.sqrt(gmm.covariances_.flatten())

# Sort components by variance (calm = lower variance)
order = np.argsort(stds)
weights = weights[order]
means = means[order]
stds = stds[order]

print("   Gaussian Mixture Model (2 components):")
print(f"   {'Component':<12} {'Weight':>8} {'Mean':>12} {'Std':>12} {'Interpretation'}")
print(f"   {'-'*60}")
print(f"   {'Calm':<12} {weights[0]:>8.4f} {means[0]:>12.6f} {stds[0]:>12.6f} Low volatility")
print(f"   {'Crisis':<12} {weights[1]:>8.4f} {means[1]:>12.6f} {stds[1]:>12.6f} High volatility")
print(f"\n   Vol ratio (crisis/calm): {stds[1]/stds[0]:.2f}x")

# --- Plot: Histogram + mixture PDF + component PDFs ---
fig, ax = plt.subplots(figsize=(7, 3.5))

ax.hist(log_ret, bins=150, density=True, alpha=0.35, color=MAIN_BLUE,
        edgecolor='none', label='S&P 500 returns')

x_grid = np.linspace(log_ret.min() * 1.1, log_ret.max() * 1.1, 500)

# Individual components
pdf_calm = weights[0] * stats.norm.pdf(x_grid, means[0], stds[0])
pdf_crisis = weights[1] * stats.norm.pdf(x_grid, means[1], stds[1])
pdf_mix = pdf_calm + pdf_crisis

ax.plot(x_grid, pdf_calm, color=FOREST, linewidth=1.0, linestyle='--',
        label=f'Calm ($\\pi$={weights[0]:.2f}, $\\sigma$={stds[0]*100:.2f}%)')
ax.plot(x_grid, pdf_crisis, color=CRIMSON, linewidth=1.0, linestyle='--',
        label=f'Crisis ($\\pi$={weights[1]:.2f}, $\\sigma$={stds[1]*100:.2f}%)')
ax.plot(x_grid, pdf_mix, color=ORANGE, linewidth=1.3,
        label='Mixture PDF')
ax.fill_between(x_grid, pdf_calm, alpha=0.08, color=FOREST)
ax.fill_between(x_grid, pdf_crisis, alpha=0.08, color=CRIMSON)

ax.set_xlabel('Log-return')
ax.set_ylabel('Density')
ax.set_title('Gaussian Mixture Model (2 components)', fontweight='bold', fontsize=9)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2,
          frameon=False, fontsize=7)

plt.tight_layout()
save_fig('ch2_gaussian_mixture')

## Regime Classification

In [ ]:
# Posterior probabilities for regime classification
probs = gmm.predict_proba(X)

# Probability of being in the crisis (high-vol) regime
# order[1] is the crisis component index in the original GMM
crisis_prob = probs[:, order[1]]
regime = (crisis_prob > 0.5).astype(int)  # 0 = calm, 1 = crisis

n_calm = np.sum(regime == 0)
n_crisis = np.sum(regime == 1)
print(f"   Regime classification:")
print(f"   Calm days:   {n_calm} ({100*n_calm/len(regime):.1f}%)")
print(f"   Crisis days: {n_crisis} ({100*n_crisis/len(regime):.1f}%)")

# --- Plot ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 5), sharex=True,
                                gridspec_kw={'height_ratios': [2, 1]})

# Panel 1: Returns colored by regime
dates = log_ret.index
calm_mask = regime == 0
crisis_mask = regime == 1

ax1.scatter(dates[calm_mask], log_ret.values[calm_mask], s=1, alpha=0.4,
            color=FOREST, label='Calm regime', edgecolors='none')
ax1.scatter(dates[crisis_mask], log_ret.values[crisis_mask], s=1, alpha=0.5,
            color=CRIMSON, label='Crisis regime', edgecolors='none')
ax1.axhline(0, color='gray', linewidth=0.3)
ax1.set_ylabel('Log-return')
ax1.set_title('S&P 500 returns classified by GMM regime', fontweight='bold', fontsize=9)
ax1.legend(loc='upper right', fontsize=7, frameon=False, markerscale=5)

# Panel 2: Crisis regime probability
ax2.fill_between(dates, 0, crisis_prob, color=CRIMSON, alpha=0.3)
ax2.plot(dates, crisis_prob, color=CRIMSON, linewidth=0.4)
ax2.axhline(0.5, color='gray', linewidth=0.5, linestyle='--')
ax2.set_ylabel('P(crisis)')
ax2.set_xlabel('Date')
ax2.set_title('Posterior probability of crisis regime', fontweight='bold', fontsize=9)
ax2.set_ylim(0, 1)

plt.tight_layout()
save_fig('ch2_regime_classification')